# Field route planning — arterials + steep-slope watch segments

Builds a suggested-route map for field teams from four ingredients:

1. **Arterial road network** from OpenStreetMap (`highway` = motorway / trunk / primary / secondary, ramps folded in)
2. **Watch segments** — the parts of those arterials within a set distance of steep slopes (≥ 30° from the Copernicus DEM): drive these slowly and scan for cracking, rockfall, debris
3. **USGS rapid landslide assessment** — the provisional triage grid from the USGS event response (extent class + road impacts per ~4 km cell)
4. **Specific sites** — landslide candidates you exported from notebooks 01/02

Roads © OpenStreetMap contributors (ODbL). **Caveat:** OSM shows the pre-earthquake network — it says nothing about current passability, and OSM completeness in Venezuela varies by area.

In [ ]:
from pathlib import Path

import geopandas as gpd
import leafmap
import pandas as pd

from geer_venezuela import (
    ATTRIBUTION,
    ROAD_COLORS,
    ROADS_ATTRIBUTION,
    asset_href,
    compute_slope,
    fetch_dem,
    fetch_roads,
    load_items,
    scenes_for,
    steep_areas,
    watch_segments,
)

DATA = (Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent) / "data"

LOCATION = "la-guaira"
BUFFER_DEG = 0.02
STEEP_DEG = 30
WATCH_DISTANCE_M = 100  # roadway within this distance of a steep slope = watch segment

items = load_items("post-event")
scenes = scenes_for(items, LOCATION)
minx, miny, maxx, maxy = scenes.total_bounds
bounds = (minx - BUFFER_DEG, miny - BUFFER_DEG, maxx + BUFFER_DEG, maxy + BUFFER_DEG)

## 1. Roads, slopes, watch segments

DEM/slope are cached from notebook 02 (computed here on first run). The Overpass query takes ~10–30 s.

In [ ]:
slope_file = compute_slope(
    fetch_dem(bounds, DATA / "terrain" / f"{LOCATION}_dem.tif"),
    DATA / "terrain" / f"{LOCATION}_slope.tif",
)
steep = steep_areas(slope_file, threshold=STEEP_DEG)

roads = fetch_roads(bounds)
watch = watch_segments(roads, steep, distance_m=WATCH_DISTANCE_M)

print(
    f"{len(roads)} arterial lines; {len(watch)} watch segments "
    f"totalling {watch['length_km'].sum():.0f} km"
)

corridors = (
    watch.dropna(subset=["name"])
    .groupby(["name", "highway"], as_index=False)["length_km"]
    .sum()
    .sort_values("length_km", ascending=False)
    .round(1)
)
corridors.head(15)

## 2. Specific sites to visit

Loads every landslide-candidate GeoJSON you have exported for this location from notebooks 01/02 (skip this cell's warning if you haven't marked any yet).

In [ ]:
candidate_files = sorted((DATA / "landslide_candidates").glob(f"{LOCATION}*.geojson"))
if candidate_files:
    sites = pd.concat(
        [gpd.read_file(f).assign(source=f.stem) for f in candidate_files], ignore_index=True
    )
    print(f"{len(sites)} candidate site(s) from {len(candidate_files)} file(s)")
else:
    sites = None
    print("No landslide candidates exported yet for this location — mark some in notebook 01/02.")

## 2b. USGS rapid landslide assessment

USGS's [2026 Venezuela Sequence landslide response](https://www.usgs.gov/programs/landslide-hazards/science/2026-venezuela-sequence-earthquake-triggered-landslide-hazards) published a provisional imagery-based triage grid on 2026-07-01 ([DOI 10.5066/P1MRLOZ7](https://doi.org/10.5066/P1MRLOZ7), cached in `data/usgs/`): each ~4 km cell is classed by landslide extent and road impact. Cells rated **major or widespread** — and any cell with road impacts **yes** — are the top route priorities.

In [ ]:
usgs = gpd.read_file(
    DATA / "usgs" / "VenezuelaEarthquake_rapid_imagery_landslide_assessment_20260701.geojson"
).cx[bounds[0] : bounds[2], bounds[1] : bounds[3]]

USGS_COLORS = {
    "major or widespread": "#d90429",
    "localized": "#f77f00",
    "minor": "#fcbf49",
    "unknown": "#adb5bd",
}

print(f"{len(usgs)} USGS assessment cells in AOI")
usgs.groupby(["landslide_exent", "road impacts"]).size().unstack(fill_value=0)

## 3. Route map

- Roads colored by class (red = motorway → blue = secondary)
- **Bold magenta = watch segments** (steep slope within 100 m)
- Candidate sites in yellow

Use the **draw tools** to sketch the actual route you want teams to drive (line), and add/adjust stops (points); export below.

In [ ]:
m = leafmap.Map()
m.add_cog_layer(
    asset_href(scenes.iloc[0], "visual"),
    name=f"AFTER — {scenes.iloc[0]['title']}",
    attribution=ATTRIBUTION,
)
m.add_gdf(
    usgs,
    layer_name="USGS rapid assessment (2026-07-01)",
    style_callback=lambda feat: {
        "fillColor": USGS_COLORS.get(feat["properties"]["landslide_exent"], "#adb5bd"),
        "fillOpacity": 0.2,
        "color": "#d90429" if feat["properties"]["road impacts"] == "yes" else "#666666",
        "weight": 2.5 if feat["properties"]["road impacts"] == "yes" else 0.5,
    },
    info_mode="on_hover",
    zoom_to_layer=False,
)
m.add_gdf(
    steep,
    layer_name=f"Steep areas (≥{STEEP_DEG}°)",
    style={"color": "#ff3b30", "weight": 1, "fillOpacity": 0.08},
    info_mode=None,
    zoom_to_layer=False,
)
m.add_gdf(
    roads,
    layer_name="Arterials (OSM)",
    style_callback=lambda feat: {
        "color": ROAD_COLORS.get(feat["properties"]["highway"], "#888888"),
        "weight": 2,
        "opacity": 0.8,
    },
    info_mode="on_hover",
    zoom_to_layer=False,
)
m.add_gdf(
    watch,
    layer_name=f"WATCH — steep slope within {WATCH_DISTANCE_M} m",
    style={"color": "#ff00ff", "weight": 5, "opacity": 0.85},
    info_mode="on_hover",
    zoom_to_layer=False,
)
if sites is not None:
    m.add_gdf(
        sites,
        layer_name="Candidate sites",
        style={"color": "#ffd60a", "weight": 3, "fillOpacity": 0.4},
        zoom_to_layer=False,
    )
m.add_legend(title="Road class", legend_dict=ROAD_COLORS, position="bottomleft")
m.add_legend(
    title="USGS landslide extent", legend_dict=USGS_COLORS, position="bottomright"
)
m

## 4. Export the route package

Everything a field team needs, as GeoJSON in `data/routes/` (opens in QGIS, ArcGIS Online, Google Earth, most GPS apps).

In [ ]:
routes_dir = DATA / "routes"
routes_dir.mkdir(parents=True, exist_ok=True)

watch.to_file(routes_dir / f"{LOCATION}_watch_segments.geojson")
roads.to_file(routes_dir / f"{LOCATION}_arterials.geojson")
corridors.to_csv(routes_dir / f"{LOCATION}_watch_corridors.csv", index=False)
print(f"Saved watch segments, arterials, corridor summary to {routes_dir}")

if m.user_rois is not None and m.user_rois["features"]:
    route_file = routes_dir / f"{LOCATION}_suggested_route.geojson"
    m.save_draw_features(str(route_file), indent=2)
    print(f"Saved drawn route ({len(m.user_rois['features'])} feature(s)) to {route_file}")
else:
    print("No route drawn yet — sketch one on the map above and re-run this cell.")

## Notes

- Watch segments are a *screening* product: a 100 m buffer around ≥30° slopes from a 30 m surface model. Cut-slope failures along benched road cuts can occur below that threshold.
- Confirm road passability locally before committing to a route — bridges and tunnels (Autopista Caracas–La Guaira viaducts!) are single points of failure.
- To tighten/loosen the screen, change `WATCH_DISTANCE_M` or `STEEP_DEG` and re-run.